# Week 3: Imbalanced Classification and LightGBM Modeling

**Day 1 - Confirm the imbalance problem**

Goal for today: load the fused dataset from Week 2, measure exactly how rare machine failures are, install the libraries you'll need for the rest of the week, and see concretely why accuracy is the wrong metric for this problem before writing a single line of modeling code.



In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import imblearn

print("lightgbm:", lgb.__version__)
print("imbalanced-learn:", imblearn.__version__)

In [ ]:
DATA_PATH = "ai4i_fused_week2.csv"  # your Week 2 output

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Step 1 - Measure exactly how rare failures are

The brief states failures are under 2% of the data - let's verify that on your actual fused dataset rather than assuming.

In [ ]:
target = "Machine failure"

counts = df[target].value_counts()
pct = df[target].value_counts(normalize=True) * 100

print(counts)
print()
print(pct.round(3))

## Step 2 - Why accuracy will lie to you

Before touching LightGBM, let's prove the point with the simplest possible "model": one that always predicts "no failure," no matter what. If this dummy model still scores high on accuracy, you know accuracy can't be trusted to judge anything for the rest of this week.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score

y = df[target]

y_train, y_test = train_test_split(y, test_size=0.2, stratify=y, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
y_pred = dummy.predict(np.zeros((len(y_test), 1)))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (failures caught): {recall_score(y_test, y_pred):.4f}")
print(f"F1 score: {f1_score(y_test, y_pred):.4f}")

## Takeaway for Day 1

A model that never predicts a failure can score over 95% accuracy while catching **zero** real failures - exactly the opposite of what a predictive maintenance system needs to do. For the rest of this week, judge every model by **recall**, **F1**, and **ROC-AUC / PR-AUC**, never accuracy alone.


**Day 2 - Build the real 5-fold stratified CV pipeline (no SMOTE yet)**

Goal: set up `StratifiedKFold`, train a baseline LightGBM with no resampling tricks, and record real recall/F1/ROC-AUC/PR-AUC numbers across the 5 folds.

### Step 0 - Watch out for a built-in leak

In the AI4I dataset, `Machine failure` is literally defined as 1 whenever *any* of the five failure-mode flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) is 1. If you leave those columns in as features, your model will "predict" failure by just reading the answer off the failure-mode columns - you'd get near-perfect recall and it would mean nothing. Drop them, along with ID/text columns that aren't real signal.

In [ ]:
target = "Machine failure"
leak_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
drop_cols = ["UDI", "Product ID", "Type", "timestamp", target] + leak_cols

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Using {len(feature_cols)} features:")
print(feature_cols)

X = df[feature_cols]
y = df[target]

### Step 1 - Why stratified matters here

With failures under 2% of rows, a plain random K-fold could easily place very few (or even zero) failures into a given fold by chance, making that fold's metrics meaningless. `StratifiedKFold` forces each fold to keep roughly the same failure rate as the full dataset. Quick check before training anything:

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    fold_rate = y.iloc[test_idx].mean() * 100
    print(f"Fold {i}: {len(test_idx)} rows, failure rate {fold_rate:.3f}%")

### Step 2 - Train a plain LightGBM per fold (the baseline)

No SMOTE, no class weighting - just LightGBM as-is. This number is what Day 3's resampling needs to beat.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score
)

def sanitize_col_names(df):
    # Replace problematic characters with underscores
    new_columns = []
    for col in df.columns:
        col = col.replace('[', '_').replace(']', '_').replace('<', '_').replace('>', '_').replace(' ', '_').replace('=', '_').replace(',', '_')
        new_columns.append(col)
    df.columns = new_columns
    return df

def run_cv(X, y, cv, model_fn):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = model_fn()
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows)

baseline_model_fn = lambda: lgb.LGBMClassifier(random_state=42, verbose=-1)

baseline_results = run_cv(X, y, cv, baseline_model_fn)
baseline_results

In [ ]:
summary = baseline_results.agg(["mean", "std"]).T
summary.columns = ["mean", "std"]
print("Baseline LightGBM (no resampling) - 5-fold CV results:")
summary.round(4)

In [ ]:
baseline_results.to_csv("week3_baseline_cv_results.csv", index=False)
print("Saved week3_baseline_cv_results.csv")

## Takeaway for Day 2

You now have a real baseline: a plain LightGBM, evaluated honestly across 5 stratified folds, with leakage columns removed. Whatever recall/F1/PR-AUC numbers you got above are the bar Day 3's SMOTE pipeline has to clear - and clear by a real margin, since SMOTE adds its own noise too.



**Day 3 - Get SMOTE into the loop correctly**

Goal: rebuild yesterday's exact pipeline, but chain `SMOTE -> LightGBM` using an `imblearn` Pipeline instead of plain LightGBM. Everything else - same folds, same model settings - stays identical, so any difference in the numbers is genuinely caused by SMOTE, not by a different setup.

### Why an `imblearn` Pipeline and not manual resampling

If you called `SMOTE().fit_resample()` yourself before the CV loop, you'd be back to the leak from the diagram earlier - synthetic points would be generated using information from rows that end up in the test fold. An `imblearn.pipeline.Pipeline` fixes this automatically: during `.fit()` it resamples only the training data passed in for that fold, and during `.predict()` / `.predict_proba()` it skips the resampling step entirely and just calls the model. You get the correct behavior without having to remember to do it yourself.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

def make_smote_pipeline():
    return ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("clf", lgb.LGBMClassifier(random_state=42, verbose=-1)),
    ])

# Reuses the same run_cv() and the same cv (StratifiedKFold, random_state=42) from Day 2,
# so the fold splits are identical - this is what makes the comparison fair.
smote_results = run_cv(X, y, cv, make_smote_pipeline)
smote_results

In [ ]:
smote_summary = smote_results.agg(["mean", "std"]).T
smote_summary.columns = ["mean", "std"]
print("SMOTE + LightGBM - 5-fold CV results:")
smote_summary.round(4)

### Step 1- Compare against yesterday's baseline, side by side

In [ ]:
baseline_results = pd.read_csv("week3_baseline_cv_results.csv")  # reload in case the kernel restarted

comparison = pd.DataFrame({
    "baseline_mean": baseline_results.mean(),
    "smote_mean": smote_results.mean(),
})
comparison["improvement"] = comparison["smote_mean"] - comparison["baseline_mean"]
comparison.round(4)

### Reading the comparison table

SMOTE usually pushes **recall** up (the model catches more real failures) but can pull **precision** down (it also raises more false alarms), because the synthetic minority samples nudge the decision boundary. Look at the whole table together, not one row in isolation:

- Recall up, F1 and PR-AUC also up -> a genuine win, keep it
- Recall up, but precision collapses and F1/PR-AUC actually got worse -> SMOTE is overfitting to synthetic points; worth revisiting on Day 4 (fewer neighbors, or combine with LightGBM's own class weighting instead of relying on SMOTE alone)

Either outcome is a valid, reportable result - the point of this exercise is to measure it honestly, not to force a win.

In [ ]:
smote_results.to_csv("week3_smote_cv_results.csv", index=False)
comparison.to_csv("week3_baseline_vs_smote_comparison.csv")
print("Saved week3_smote_cv_results.csv and week3_baseline_vs_smote_comparison.csv")